## Preparando la base

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import math
import random


from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('fivethirtyeight')

import statsmodels.api as sm
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.tsaplots import plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

from math import sqrt

import matplotlib
matplotlib.rcParams['axes.labelsize'] = 14
matplotlib.rcParams['xtick.labelsize'] = 12
matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['text.color'] = 'k'
import seaborn as sns

In [ ]:
base = pd.read_excel("Base_demanda_atunes.xlsx")

In [ ]:
base['fecha'] = pd.to_datetime(base['fecha'],format = "%d/%m/%Y")

In [ ]:
base[['fecha','Vta_Uds']].set_index('fecha')

In [ ]:
base_lima = base[['fecha','Vta_Uds']].set_index('fecha')

In [ ]:
#base_lima = base_lima.set_index('fecha')
base_lima = base_lima.resample('W').sum()

In [ ]:
base_lima

In [ ]:
plt.figure(figsize=(12,4))
sns.lineplot(data = base_lima, x = "fecha", y = "Vta_Uds", color = "blue")
plt.show()

In [ ]:
base_lima = base[base['fecha']>='2020-07-01'][['fecha','Vta_Uds']].groupby('fecha').sum()

In [ ]:
#base_lima = base_lima.set_index('fecha')
base_lima = base_lima.resample('W').sum()

In [ ]:
#base_lima=base_lima.reset_index()
B1FIN_prom =base_lima.copy()

In [ ]:
# # corrigiendo fechas que no tienen dato
# r = pd.date_range(start=B1FIN_prom.index.min(), end=B1FIN_prom.index.max())
# B1FIN_prom = B1FIN_prom.reindex(r).fillna(0.0).rename_axis('fecha').reset_index().set_index('fecha')
# B1FIN_prom[B1FIN_prom.Vta_Uds == 0] = B1FIN_prom.mean()
# decomposition = sm.tsa.seasonal_decompose(B1FIN_prom, model='multiplicative')
# matplotlib.rcParams['figure.figsize'] = 18, 8
# fig = decomposition.plot()
# plt.show()

In [ ]:
plt.figure(figsize=(12,4))
sns.lineplot(data = B1FIN_prom, x = "fecha", y = "Vta_Uds", color = "blue")
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range = (0,1))

In [ ]:
#divide into train and validation set
train = B1FIN_prom.drop(columns = [])[:int(0.80*(len(B1FIN_prom)))]
valid = B1FIN_prom[int(0.80*(len(B1FIN_prom))):]

#plotting the data
train.plot()
valid.plot()
plt.show()

In [ ]:
B1FIN_prom

In [ ]:
scaler.fit(train)
scaled_train_data = scaler.transform(np.array(train).reshape(-1,1))
scaled_test_data = scaler.transform(np.array(valid).reshape(-1,1))

In [ ]:
df1 = scaler.transform(np.array(B1FIN_prom).reshape(-1,1))

In [ ]:
len(scaled_train_data),len(scaled_test_data)

In [ ]:
import numpy
# convert an array of values into a dataset matrix
def create_dataset(dataset, time_step=1):
	dataX, dataY = [], []
	for i in range(len(dataset)-time_step-1):
		a = dataset[i:(i+time_step), 0]   ###i=0, 0,1,2,3-----99   100
		dataX.append(a)
		dataY.append(dataset[i + time_step, 0])
	return numpy.array(dataX), numpy.array(dataY)

In [ ]:
# reshape into X=t,t+1,t+2,t+3 and Y=t+4
time_step = 14
X_train, y_train = create_dataset(scaled_train_data, time_step)
X_test, y_test = create_dataset(scaled_test_data, time_step)

In [ ]:
X_train

In [ ]:
print(X_train.shape), print(y_train.shape)

In [ ]:
print(X_test.shape), print(y_test.shape)

In [ ]:
# reshape input to be [samples, time steps, features] which is required for LSTM
X_train =X_train.reshape(X_train.shape[0],X_train.shape[1] , 1)
X_test = X_test.reshape(X_test.shape[0],X_test.shape[1] , 1)

In [ ]:
X_train.shape

## <a id='5.2.'>5.2. LSTM Model </a>

In [ ]:
def genera_modelo(X_train,y_train,X_test, ytest):
    epochs = [30]  #epocas
    ranks = [50]    # nodos
    min_error = float('inf')
    best_model = None
    for rank in ranks:
      pred_train = []
      lista_train = []
      model=Sequential()
      model.add(LSTM(rank,return_sequences=True,input_shape=(14,1)))
      model.add(LSTM(rank,return_sequences=True))
      model.add(LSTM(rank))
      model.add(Dense(1))
      model.compile(loss='mean_squared_error',optimizer='adam')
      for epoch in epochs:
        model.fit(X_train,y_train,validation_data=(X_test,ytest),epochs=epoch,batch_size=64,verbose=1)
        train_predict=model.predict(X_train)
        test_predict=model.predict(X_test)
        ##Transformback to original form
        train_predict=scaler.inverse_transform(train_predict)
        test_predict=scaler.inverse_transform(test_predict)

        calrmse_train = sqrt(mean_squared_error(y_train,train_predict))
        calrmse_test = sqrt(mean_squared_error(ytest,test_predict))

        indicador=abs(calrmse_train-calrmse_test)*calrmse_train

        if indicador < min_error:
          min_error = indicador
          best_epochs = epoch
          best_dens = rank
          best_model=model

    return(best_model,calrmse_train,calrmse_test,train_predict,test_predict,best_epochs)

In [ ]:
best_model,calrmse_train,calrmse_test,train_predict,test_predict,b=genera_modelo(X_train,y_train,X_test, y_test)

In [ ]:
best_model.summary()

In [ ]:
### Plotting
# shift train predictions for plotting
look_back=14
trainPredictPlot = numpy.empty_like(df1)
trainPredictPlot[:, :] = np.nan
trainPredictPlot[look_back:len(train_predict)+look_back, :] = train_predict
# shift test predictions for plotting
testPredictPlot = numpy.empty_like(df1)
testPredictPlot[:, :] = numpy.nan
testPredictPlot[len(train_predict)+(look_back*2)+1:len(df1)-1, :] = test_predict

plt.figure(figsize=(10, 7))
# plot baseline and predictions
plt.plot(scaler.inverse_transform(df1[:]))
plt.plot(trainPredictPlot[:])
plt.plot(testPredictPlot[:])

#plt.plot(scaler.inverse_transform(df1))
#plt.plot(trainPredictPlot)
#plt.plot(testPredictPlot)

plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, median_absolute_error, mean_absolute_percentage_error

In [ ]:
def evaluate_forecast(y,pred):
    results = pd.DataFrame({'r2_score':r2_score(y, pred),
                           }, index=[0])
    results['mean_absolute_error'] = mean_absolute_error(y, pred)
    results['median_absolute_error'] = median_absolute_error(y, pred)
    results['mse'] = mean_squared_error(y, pred)
    #results['msle'] = mean_squared_log_error(y, pred)
    results['rmse'] = np.sqrt(results['mse'])
    results['mape'] = mean_absolute_percentage_error(y, pred)
    return results

In [ ]:
evaluate_forecast(train[15:], train_predict)

In [ ]:
evaluate_forecast(valid[15:], test_predict)

In [ ]:
test_predict